# "THE POUR IS RIGHT" - Week 7, Colab notebook 2 of 3

# Fine-tune the model with QLoRA

This is the main event: we train `Qwen/Qwen3-4B-Base` on our 20,000 wines. It mirrors Ed's week 7 days 3 and 4.

## What QLoRA is

Training all 4 billion numbers in the model would need far more memory than a free GPU has. QLoRA gets round that in two ways:

- **Q for quantized.** The base model is loaded at 4 bits per number and then frozen. It is never changed.
- **LoRA for Low-Rank Adaptation.** We attach small new layers, called adapters, alongside parts of the model, and train only those. They hold a few tens of millions of numbers instead of 4 billion.

The result is a small adapter file (a hundred megabytes or so) that sits on top of the unchanged base model.

## Time

Training runs for one pass over the 20,000 wines. On a free T4 it takes about an hour and a half. Progress is saved to your Hugging Face account every 500 steps, so a disconnect does not lose everything.

Keep the browser tab open while it runs.

## Before you start

1. **Use a GPU.** In the menu: Runtime -> Change runtime type -> T4 GPU -> Save.
2. **Add your Hugging Face token as a secret.** Click the key icon in the left sidebar, add a secret named `HF_TOKEN` with your token as the value, and switch on "Notebook access".

Then run the cells in order.

In [ ]:
# Install the libraries. The versions are pinned to the ones the training code was tested against.
# This takes a minute or two. Warnings about other packages' dependencies can be ignored.

!pip install -q --upgrade transformers==5.18.0 trl==1.14.1 peft==0.21.2 accelerate bitsandbytes datasets

In [ ]:
import torch
from datetime import datetime
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [ ]:
# Log in to Hugging Face with the token saved in Colab's secrets

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Check that the GPU is there

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Using {gpu} with {memory:.0f} GB of memory")
else:
    print("No GPU found. Go to Runtime -> Change runtime type and choose T4 GPU, then start again.")

## Constants and hyper-parameters

A hyper-parameter is a setting we choose before training, as opposed to the numbers the training itself learns. These are Ed's values where I know them, adjusted for our data.

**The run name** is stamped with the date and time, and becomes part of the model's name on Hugging Face. The cell prints it: **copy it down**, because notebook 3 needs it.

**LoRA settings**

- `LORA_R = 32`: the size of the adapters. Bigger means more capacity to learn, more memory, and more risk of memorising.
- `LORA_ALPHA = 64`: how strongly the adapters' output is applied. The usual rule is twice R.
- `LORA_DROPOUT = 0.1`: switches off a random 10% of adapter values at each step, to reduce memorising. The same idea as in the deep neural network on day 4.
- `TARGET_MODULES`: which parts of the model get adapters. These four are the attention layers, where the model decides which words to pay attention to.

**Training settings**

- `EPOCHS = 1`: one pass over the 20,000 wines.
- `BATCH_SIZE = 8`: wines per step. 20,000 / 8 = 2,500 steps. If you get an out-of-memory error, change it to 4.
- `LEARNING_RATE = 1e-4`: how big each adjustment is.
- `LR_SCHEDULER_TYPE = "cosine"` and `WARMUP = 0.03`: the learning rate ramps up over the first 3% of steps, then falls smoothly to zero.
- `MAX_LENGTH = 192`: the longest text allowed, in tokens. Our longest is 172.

In [ ]:
# Constants

HF_USER = "Warumanu"
DATASET_NAME = f"{HF_USER}/wines_prompts_lite"
BASE_MODEL = "Qwen/Qwen3-4B-Base"
PROJECT_NAME = "wine-pricer"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# LoRA hyper-parameters

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# Training hyper-parameters

EPOCHS = 1
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = "cosine"
WARMUP = 0.03
WEIGHT_DECAY = 0.001
MAX_GRAD_NORM = 0.3
OPTIMIZER = "paged_adamw_32bit"
MAX_LENGTH = 192

# How often to report, check and save

LOG_STEPS = 50
SAVE_STEPS = 500

print(f"This run will be saved as: {HUB_MODEL_NAME}")

## The data

We train on all 20,000 training wines. During training the model is also checked on 200 validation wines it never trains on, so we can see whether it is learning or just memorising.

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
val = dataset["val"].select(range(200))
test = dataset["test"]

print(f"{len(train):,} training wines, {len(val):,} validation wines used during training")

## Load the base model in 4 bits

The same loading as in notebook 1.

In [ ]:
# Quantization: load the model's 4 billion numbers at 4 bits each instead of 16 or 32

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

In [ ]:
# Load the tokenizer and the base model. The first run downloads about 8 GB, which takes a few minutes

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

## Set up the training

Two configuration objects: one for the adapters, one for the training run.

The training library, TRL, sees that our dataset has `prompt` and `completion` columns and handles the rest. It joins them, adds an end-of-text marker, and computes the loss on the completion only. In other words the model is graded on the price, not on reproducing the wine description. I checked this on a small test: for a 113-token wine, only the six price tokens counted.

### Number formats on a T4

Two settings here are specific to the free T4 GPU, and both were found the hard way:

- The model is loaded with `dtype=torch.float16` and trained with `fp16=True`. The T4 handles the float16 format natively. With the newer bfloat16 format (`bf16=True`) the same run was six times slower, because the T4 has to emulate it.
- The cell after the trainer puts the adapter weights back into full 32-bit precision. TRL stores them in bfloat16 for a 4-bit model, and `fp16=True` then fails on the first step.

On a newer GPU (L4, A100) you would do the opposite: remove the `dtype` line and the extra cell, and use `bf16=True`.

In [ ]:
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=WARMUP,
    train_sampling_strategy="group_by_length",
    fp16=True,
    max_length=MAX_LENGTH,
    logging_steps=LOG_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to="none",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_strategy="every_save",
    hub_private_repo=True,
)

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
    processing_class=tokenizer,
)

In [ ]:
# Keep the adapter weights in full 32-bit precision, so they work with fp16 training on a T4

for param in fine_tuning.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

In [ ]:
# How many numbers are we actually training?

fine_tuning.model.print_trainable_parameters()

## Train

**This is the long cell.** A progress bar shows the step out of 2,500 and the time remaining, about an hour and a half. Every 500 steps it pauses to check the validation wines, saves a checkpoint, and adds a row to the table with the training loss and the validation loss. The table stays empty until step 500.

What to look for:

- **Training loss falling** from its starting value: the model is learning.
- **Validation loss falling too**: it is learning something that carries over to wines it has not seen. If the training loss keeps falling while the validation loss rises, it has started memorising.

If you see `CUDA out of memory`, go to Runtime -> Restart session, change `BATCH_SIZE` to 4 in the constants cell, and run everything again.

In [ ]:
fine_tuning.train()

## Save the final model to the Hub

Checkpoints were pushed along the way. This pushes the finished adapter. It is saved as a private model on your account.

In [ ]:
fine_tuning.model.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Saved to the hub: {HUB_MODEL_NAME}")

## A quick look before you leave

The trained model is still in memory, so we can ask it for a few prices right away. The full evaluation is in notebook 3.

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuning.model.generate(**inputs, max_new_tokens=8)
    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(output_ids[0, prompt_length:], skip_special_tokens=True)

fine_tuning.model.eval()
for i in range(5):
    print(f"Model says: {model_predict(test[i])!r}   Real price: {test[i]['completion']}")

## What to note down

The model name printed above, for example `Warumanu/wine-pricer-2026-10-06_14.30.00`. Notebook 3 needs it.

## If Colab disconnected part-way

Checkpoints are on your Hugging Face account under the run name printed by the constants cell. Notebook 3 can evaluate the latest checkpoint as it is: a model that got through, say, 1,500 of 2,500 steps is still a trained model.